### <Center> Лабораторна робота №7. <br> Алгоритм випадкового лісу в задачі кредитного скорінгу

Необхідно розв'язати задачу кредитного скорінга 

Ознаки клієнта банку:
- Age - вік (дійсночислова)
- Income - місячний дохід (дійсночислова)
- BalanceToCreditLimit - відношення балансу на кредитній картці до ліміту за кредитом (дійсночислова)
- DIR - Debt-to-income Ratio (дійсночислова)
- NumLoans - кылькість позичок і кредитних ліній
- NumRealEstateLoans - кількість іпотек і позичок, пов'язаних з нерухомістю (натуральне число)
- NumDependents - кількість членів сім'ї, яких утримує клієнт, без врахування самого клієнту (натуральне число)
- Num30-59Delinquencies - кількість протермінувань виплат за кредитом від 30 до 59 днів (натуральне число)
- Num60-89Delinquencies - кількість протермінувань виплат за кредитом від 60 до 89 дній (натуральне число)
- Delinquent90 - чи були протермінування виплат за кредитом більше 90 днів (бінарний)

In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
# %matplotlib inline

**Спочатку налаштуємо доступ до даних на google drive (якщо ви відкриваєте блокнот в google colab, а не на PC) шляхом монтування google drive**

In [2]:
# Для локального запуску Google Drive не потрібен.

Перевіримо шлях до папки з матеріалами лаборатоної роботи на google drive. Якщо у вас шлях відрізняється то відредагуйте

In [3]:
# Дані завантажуються з локальної папки lab7/data.

Перемістимо матеріали лабораторної роботи з google drive на віртуальну машину google colab

In [4]:
# Копіювання з Google Drive не потрібне під час локального запуску.

Завантажимо дані з використанням pandas

In [5]:
from pathlib import Path
data_folder = Path('data')
if not (data_folder / 'credit_scoring_train.csv').exists():
    data_folder = Path('lab7/data')
output_folder = data_folder.parent / 'results'
output_folder.mkdir(exist_ok=True)
train_df = pd.read_csv(data_folder / 'credit_scoring_train.csv', index_col='client_id')
test_df = pd.read_csv(data_folder / 'credit_scoring_test.csv', index_col='client_id')

In [6]:
y = train_df['Delinquent90']
train_df.drop('Delinquent90', axis=1, inplace=True)

In [7]:
train_df.head()

,DIR,Age,NumLoans,NumRealEstateLoans,NumDependents,Num30-59Delinquencies,Num60-89Delinquencies,Income,BalanceToCreditLimit
client_id,,,,,,,,,
0,0.496289,49.1,13,0,0.0,2,0,5298.360639,0.387028
1,0.433567,48.0,9,2,2.0,1,0,6008.056256,0.234679
2,2206.731199,55.5,21,1,NaN,1,0,NaN,0.348227
3,886.132793,55.3,3,0,0.0,0,0,NaN,0.971930
4,0.000000,52.3,1,0,0.0,0,0,2504.613105,1.004350


**Переглянемо кількість пропусків в кожній ознаці.**

In [8]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75000 entries, 0 to 74999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   DIR                    75000 non-null  float64
 1   Age                    75000 non-null  float64
 2   NumLoans               75000 non-null  int64  
 3   NumRealEstateLoans     75000 non-null  int64  
 4   NumDependents          73084 non-null  float64
 5   Num30-59Delinquencies  75000 non-null  int64  
 6   Num60-89Delinquencies  75000 non-null  int64  
 7   Income                 60153 non-null  float64
 8   BalanceToCreditLimit   75000 non-null  float64
dtypes: float64(5), int64(4)
memory usage: 5.7 MB


In [9]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75000 entries, 75000 to 149999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   DIR                    75000 non-null  float64
 1   Age                    75000 non-null  float64
 2   NumLoans               75000 non-null  int64  
 3   NumRealEstateLoans     75000 non-null  int64  
 4   NumDependents          72992 non-null  float64
 5   Num30-59Delinquencies  75000 non-null  int64  
 6   Num60-89Delinquencies  75000 non-null  int64  
 7   Income                 60116 non-null  float64
 8   BalanceToCreditLimit   75000 non-null  float64
dtypes: float64(5), int64(4)
memory usage: 5.7 MB


**Замінимо пропуски медіанними значеннями.**

In [10]:
train_df['NumDependents'] = train_df['NumDependents'].fillna(train_df['NumDependents'].median())
train_df['Income'] = train_df['Income'].fillna(train_df['Income'].median())
test_df['NumDependents'] = test_df['NumDependents'].fillna(train_df['NumDependents'].median())
test_df['Income'] = test_df['Income'].fillna(train_df['Income'].median())

### Дерево рішень без налаштування параметрів

**Обучите дерево решений максимальной глубины 3, используйте параметр random_state=17 для воспроизводимости результатов.**

In [11]:
first_tree = DecisionTreeClassifier(max_depth=3, random_state=17)
first_tree.fit(train_df, y)
print('Дерево рішень навчено. Максимальна глибина:', first_tree.max_depth)

Дерево рішень навчено. Максимальна глибина: 3


**Зробіть прогноз для тестової вибірки.**

In [12]:
first_tree_pred = first_tree.predict(test_df)
print('Кількість прогнозів дерева рішень:', len(first_tree_pred))

Кількість прогнозів дерева рішень: 75000


**Запишемо прогноз у файл.**

In [13]:
def write_to_submission_file(predicted_labels, out_file,
                             target='Delinquent90', index_label="client_id"):
    # turn predictions into data frame and save as csv file
    predicted_df = pd.DataFrame(predicted_labels,
                                index = np.arange(75000, 
                                                  predicted_labels.shape[0] + 75000),
                                columns=[target])
    predicted_df.to_csv(out_file, index_label=index_label)

In [14]:
write_to_submission_file(first_tree_pred, output_folder / 'credit_scoring_first_tree.csv')
print('Файл створено:', output_folder / 'credit_scoring_first_tree.csv')

Файл створено: results/credit_scoring_first_tree.csv


**Якщо прогнозувати ймовірності дефолту для клієнтів тестової вибірки, результат буде набагато кращим.**

In [15]:
first_tree_pred_probs = first_tree.predict_proba(test_df)[:, 1]

In [16]:
write_to_submission_file(first_tree_pred_probs, output_folder / 'credit_scoring_first_tree_probs.csv')
print('Файл створено:', output_folder / 'credit_scoring_first_tree_probs.csv')

Файл створено: results/credit_scoring_first_tree_probs.csv


## Дерево рішень без налаштування параметрів за допомогою GridSearch

**Налаштуйте параметри дерева за допомогою `GridSearhCV`, подивіться на кращу комбінацію параметрів і середню якість на 5-кратній крос-валідації. Використовуйте параметр `random_state=17` (для відтворюваності результатів), не забувайте про розпаралелювання (`n_jobs=-1`).**

In [17]:
tree_params = {'max_depth': list(range(3, 8)), 
               'min_samples_leaf': list(range(5, 13))}

locally_best_tree = GridSearchCV(DecisionTreeClassifier(random_state=17), tree_params, cv=5, scoring='roc_auc', n_jobs=-1)
locally_best_tree.fit(train_df, y)
print('Найкращі параметри дерева:', locally_best_tree.best_params_)
print('Найкращий ROC AUC дерева:', round(locally_best_tree.best_score_, 4))

Найкращі параметри дерева: {'max_depth': 6, 'min_samples_leaf': 12}
Найкращий ROC AUC дерева: 0.8288


In [18]:
print('Параметри дерева:', locally_best_tree.best_params_)
print('Середній ROC AUC на крос-валідації:', round(locally_best_tree.best_score_, 3))

Параметри дерева: {'max_depth': 6, 'min_samples_leaf': 12}
Середній ROC AUC на крос-валідації: 0.829


**Зробіть прогноз для тестової вибірки.**

In [19]:
tuned_tree_pred_probs = locally_best_tree.predict_proba(test_df)[:, 1]

In [20]:
write_to_submission_file(tuned_tree_pred_probs, output_folder / 'credit_scoring_tuned_tree.csv')
print('Файл створено:', output_folder / 'credit_scoring_tuned_tree.csv')

Файл створено: results/credit_scoring_tuned_tree.csv


### Випадковий ліс без настройки параметрів

**Навчіть випадковий ліс з дерев необмеженої глибини, використовуйте параметр `random_state=17` для відтворюваності результатів.**

In [21]:
first_forest = RandomForestClassifier(random_state=17, n_jobs=-1)
first_forest.fit(train_df, y)
print('Випадковий ліс навчено. Кількість дерев:', first_forest.n_estimators)

Випадковий ліс навчено. Кількість дерев: 100


In [22]:
first_forest_pred = first_forest.predict_proba(test_df)[:, 1]

**Зробіть прогноз для тестової вибірки.**

In [23]:
write_to_submission_file(first_forest_pred, output_folder / 'credit_scoring_first_forest.csv')
print('Файл створено:', output_folder / 'credit_scoring_first_forest.csv')

Файл створено: results/credit_scoring_first_forest.csv


### Випадковий ліс з налаштуванням параметрів

**Налаштуйте параметр `max_features` лісу за допомогою `GridSearhCV`, подивіться на кращу комбінацію параметрів і середню якість на 5-кратній крос-валідації. Використовуйте параметр random_state=17 (для відтворюваності результатів), не забувайте про розпаралелювання (n_jobs=-1).**

In [24]:
%%time
forest_params = {'max_features': np.linspace(.3, 1, 7)}

locally_best_forest = GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=17, n_jobs=-1), forest_params, cv=5, scoring='roc_auc', n_jobs=-1)
locally_best_forest.fit(train_df, y)
print('Найкращі параметри випадкового лісу:', locally_best_forest.best_params_)
print('Найкращий ROC AUC лісу:', round(locally_best_forest.best_score_, 4))

Найкращі параметри випадкового лісу: {'max_features': np.float64(0.3)}
Найкращий ROC AUC лісу: 0.8241
CPU times: user 11.9 s, sys: 244 ms, total: 12.1 s
Wall time: 1min 4s


In [25]:
print('Параметри випадкового лісу:', locally_best_forest.best_params_)
print('Середній ROC AUC на крос-валідації:', round(locally_best_forest.best_score_, 3))

Параметри випадкового лісу: {'max_features': np.float64(0.3)}
Середній ROC AUC на крос-валідації: 0.824


In [26]:
tuned_forest_pred = locally_best_forest.predict_proba(test_df)[:, 1]

In [27]:
write_to_submission_file(tuned_forest_pred, output_folder / 'credit_scoring_tuned_forest.csv')
print('Файл створено:', output_folder / 'credit_scoring_tuned_forest.csv')

Файл створено: results/credit_scoring_tuned_forest.csv


**Подивіться, як налаштований випадковий ліс оцінює важливість ознак за їх впливом на цільову ознаку. Подайте результати в наглядному вигляді за допомогою `DataFrame`.**

In [28]:
feature_importance = pd.DataFrame({'feature': train_df.columns, 'importance': locally_best_forest.best_estimator_.feature_importances_}).sort_values('importance', ascending=False).reset_index(drop=True)
print('Важливість ознак:')
display(feature_importance)

Важливість ознак:


,feature,importance
0,BalanceToCreditLimit,0.225870
1,DIR,0.170165
2,Age,0.160547
3,Income,0.149931
4,NumLoans,0.093448
5,Num60-89Delinquencies,0.065612
6,Num30-59Delinquencies,0.063031
7,NumDependents,0.038538
8,NumRealEstateLoans,0.032858


**Як правило збільшення кількості дерев тільки покращує результат. Так що на останок навчіть випадковий ліс з 300 дерев зі знайденими кращими параметрами. Це може зайняти декілька хвилин.**

In [29]:
%%time
final_forest = RandomForestClassifier(n_estimators=300, max_features=locally_best_forest.best_params_['max_features'], random_state=17, n_jobs=-1)
final_forest.fit(train_df, y)
final_forest_pred = final_forest.predict_proba(test_df)[:, 1]
write_to_submission_file(final_forest_pred, output_folder / 'credit_scoring_final_forest.csv')
print('Файл створено:', output_folder / 'credit_scoring_final_forest.csv')

Файл створено: results/credit_scoring_final_forest.csv
CPU times: user 34.5 s, sys: 330 ms, total: 34.8 s
Wall time: 4.03 s
